# NLA Studio on Colab (qwen7b)

Runs the released **kitft Qwen2.5-7B NLA** end to end: base-model extraction, the AV (vector to text) on **SGLang/GPU**, the AR (text to vector) on **CPU**. The code is pulled with `git clone` from `nla-studio` on your fork; the only file you manage is this notebook.

**Set the runtime first:** Runtime -> Change runtime type -> GPU **A100** (Pro+) or **L4** (Pro), shape **High-RAM**.
The free **T4 will not fit**. qwen7b is ungated, so no token is needed.

Run the cells top to bottom. Cell 5b self-tests the full round-trip and prints PASS/FAIL.


In [ ]:
!nvidia-smi


## 1. Get the code (git clone)
Re-run to pull the latest after any push.


In [ ]:
!rm -rf /content/natural_language_autoencoders
!git clone --depth 1 -b nla-studio https://github.com/agastyasridharan/natural_language_autoencoders.git /content/natural_language_autoencoders
%cd /content/natural_language_autoencoders/nla_studio
!ls


## 2. Install dependencies (~5 min)
Pinned stack: `sglang[all]==0.5.6` + `transformers==4.57.1` + `torch==2.9.1`.


In [ ]:
!pip install -q -r requirements.txt


## 3. Preflight: verify the tokenizer stack (seconds, no GPU)
Runs the exact gate `/load_family` uses. If it **FAILs** with "injection token appears 0x", transformers
got upgraded; run the printed fix, then **Runtime -> Restart session** and re-run from cell 2.


In [ ]:
import transformers, tokenizers
print("transformers", transformers.__version__, "| tokenizers", tokenizers.__version__)
!python scripts/preflight_tokenizer.py --family qwen7b


## 4. Launch the AV (SGLang) server on the GPU
Downloads the 7B AV (~15 GB), loads it, waits until healthy.


In [ ]:
import subprocess, time, os, httpx
sg = subprocess.Popen(["bash", "scripts/launch_sglang.sh", "qwen7b"],
                      stdout=open("sglang.log", "w"), stderr=subprocess.STDOUT)
ok = False
for _ in range(240):                      # ~20 min cap (download + load)
    try:
        httpx.get("http://localhost:30000/get_model_info", timeout=5); ok = True; break
    except Exception:
        if sg.poll() is not None:
            print("SGLang exited early - see log below"); break
        time.sleep(5)
print("AV up" if ok else "AV NOT up")
!tail -n 20 sglang.log


## 5. Start the web app (base + AR on CPU)
`NLA_INPROC_DEVICE=cpu` keeps the single GPU free for the AV.


In [ ]:
os.environ["NLA_INPROC_DEVICE"] = "cpu"
os.environ["NLA_SGLANG_URL"]    = "http://localhost:30000"
uv = subprocess.Popen(["python", "-m", "uvicorn", "app.server:app", "--host", "0.0.0.0", "--port", "8000"],
                      stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
!tail -n 20 uvicorn.log


## 5b. Self-test the full round-trip (recommended)
Loads qwen7b into the app (base+AR on CPU, ~1-3 min) and runs extract -> AV -> AR-score through the live
server. **PASS** means injection works (English, not CJK). This also leaves the family loaded, so the UI
below is ready to Extract immediately.


In [ ]:
import httpx
BASE, TEXT = "http://localhost:8000", "The quick brown fox jumps over the lazy dog."
print("Loading qwen7b (base+AR on CPU; first run also downloads ~26 GB, so allow time)…")
info = httpx.post(f"{BASE}/load_family", json={"family":"qwen7b"}, timeout=3600).json()
print("  ", {k: info.get(k) for k in ("d_model","layer_k","hidden_state_index","injection_scale","mse_scale")})
ext = httpx.post(f"{BASE}/extract", json={"text":TEXT}, timeout=600).json()
idx = ext["suggested_index"]; print(f"  {ext['n_tokens']} tokens; round-tripping position {idx}")
rt = httpx.post(f"{BASE}/roundtrip", json={"text":TEXT,"token_index":idx,"temperature":0.7}, timeout=600).json()
print("\nAV explanation:\n  " + rt["explanation"].replace("\n","\n  "))
print(f"\ncosine={rt['cosine']}  MSE={rt['mse']}  CJK_fraction={rt['cjk_fraction']}")
print("\n✅ PASS: English AV output, full round-trip works." if not rt["looks_like_injection_failure"]
      else "\n❌ FAIL: AV output mostly CJK — injection failed (check sglang Gemma patch / version pins).")


## 6. Open the UI
Family is already loaded from 5b, so go straight to **Extract -> Explain -> Score** (or click Load to reload).


In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)


## Troubleshooting
- **"injection token appears 0x"**: transformers got upgraded past 4.57.x. Run
  `!pip install "sglang[all]==0.5.6" "transformers==4.57.1"`, **Runtime -> Restart session**, re-run from cell 2.
- **pip fails on `flashinfer` / `sgl-kernel`**: those are CUDA wheels (pinned flashinfer==0.5.3, sgl-kernel==0.3.18.post2).
  Make sure a **GPU runtime** is selected before cell 2 so CUDA wheels resolve; if it still fails, retry
  `!pip install -q "sglang[all]==0.5.6"` once on its own (transient index issues are common).
- **OOM on the GPU**: kill SGLang, relaunch smaller: `!NLA_MEM_FRACTION=0.7 bash scripts/launch_sglang.sh qwen7b`.
- **Extract is slow**: 7B forward on CPU (~10-60 s per token click); keep text short, use the suggested token.
- **gemma / llama**: set `os.environ["HF_TOKEN"]="hf_..."` before cells 2-4 (gated). **llama70b will not fit one Colab GPU.**
- Logs on demand: run the cell below.


In [ ]:
!tail -n 60 sglang.log
print("\n----- uvicorn -----")
!tail -n 60 uvicorn.log
